# 第20章 文本、日期与特征处理

结合字符串、日期和数值列构造可分析的业务特征。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 批量清洗文本列
- 解析和拆解日期
- 计算时间差
- 构造分类与数值特征


## 核心概念

- Pandas字符串方法通过str访问器调用。
- 日期必须先转换为datetime才能进行时间运算。
- 特征应由业务问题驱动，并避免使用未来信息。


## 示例 1：文本标准化

清洗步骤包括去空格、统一大小写和提取模式。


In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "name": [" 张三 ", "LI SI", "王 五"],
    "email": ["A@EXAMPLE.COM", "li@test.cn", "wang@example.com"],
})
customers["name_clean"] = customers["name"].str.strip().str.replace(" ", "", regex=False)
customers["email_clean"] = customers["email"].str.strip().str.lower()
customers["domain"] = customers["email_clean"].str.extract(r"@(.+)$", expand=False)
print(customers)


## 示例 2：日期解析与拆解

errors='coerce'把无效日期转换为NaT。


In [ ]:
orders = pd.DataFrame({
    "order_date": ["2026-01-05", "2026-02-18", "invalid", "2026-03-22"],
    "amount": [320, 880, 460, 1250],
})
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["month"] = orders["order_date"].dt.to_period("M").astype("string")
orders["weekday"] = orders["order_date"].dt.day_name()
print(orders)


## 示例 3：特征构造

特征应有明确口径并可从原始字段复算。


In [ ]:
reference_date = pd.Timestamp("2026-04-01")
orders["days_ago"] = (reference_date - orders["order_date"]).dt.days
orders["amount_level"] = pd.cut(
    orders["amount"],
    bins=[0, 500, 1000, float("inf")],
    labels=["普通", "重点", "大额"],
)
print(orders)


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第20章 文本、日期与特征处理”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征


## 综合练习

1. 清洗手机号中的空格和连字符
2. 解析注册日期
3. 构造注册月份和账户天数

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
import pandas as pd

users = pd.DataFrame({
    "phone": ["138-0000-1234", " 139 0000 5678 "],
    "registered_at": ["2025-12-15", "2026-02-08"],
})

# TODO: 清洗手机号，去除所有非数字字符
users["phone_clean"] = users["phone"].str.replace(r"\D", "", regex=True)

# TODO: 解析注册日期
users["registered_at"] = pd.to_datetime(users["registered_at"])

# TODO: 提取注册月份
users["register_month"] = users["registered_at"].dt.month

# TODO: 计算账户天数（以2026-04-01为参考日期）
users["account_days"] = (pd.Timestamp("2026-04-01") - users["registered_at"]).dt.days

print(users)


In [ ]:
import pandas as pd

users = pd.DataFrame({
    "phone": ["138-0000-1234", " 139 0000 5678 "],
    "registered_at": ["2025-12-15", "2026-02-08"],
})
users["phone_clean"] = users["phone"].str.replace(r"\D", "", regex=True)
users["registered_at"] = pd.to_datetime(users["registered_at"])
users["register_month"] = users["registered_at"].dt.to_period("M").astype("string")
users["account_days"] = (pd.Timestamp("2026-04-01") - users["registered_at"]).dt.days
print(users)

# 自检
assert users["phone_clean"].tolist() == ["13800001234", "13900005678"], "检查手机号清洗"
assert users["account_days"].tolist() == [107, 52], "检查账户天数计算"


## 本章小结

结合字符串、日期和数值列构造可分析的业务特征。

**迁移思考**：

1. 如果需要提取用户邮箱的用户名部分（@符号之前），正则表达式应该如何写？
2. 为什么特征构造要避免使用未来信息？举一个会导致数据泄漏的例子。


### 你已经掌握

- 批量清洗文本列
- 解析和拆解日期
- 计算时间差
- 构造分类与数值特征


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 文本标准化 | 清洗步骤包括去空格、统一大小写和提取模式。 | `pd.DataFrame()`、`str.strip()`、`str.replace()`、`str.lower()` |
| 日期解析与拆解 | errors='coerce'把无效日期转换为NaT。 | `pd.DataFrame()`、`pd.to_datetime()`、`dt.to_period()`、`dt.day_name()` |
| 特征构造 | 特征应有明确口径并可从原始字段复算。 | `pd.Timestamp()`、`pd.cut()`、`orders["days_ago"]`、`orders["order_date"]` |


### 需要注意

- 直接对object列使用日期运算
- 正则提取失败后不检查缺失
- 使用结果变量构造导致数据泄漏的特征


### 完成检查

- [ ] 能够批量清洗文本列
- [ ] 能够解析和拆解日期
- [ ] 能够计算时间差
- [ ] 能够构造分类与数值特征
